# Food Delivery Time Prediction Project
Complete EDA, Feature Engineering, Linear Regression, and Logistic Regression workflow.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
)

df = pd.read_csv("Food_Delivery_Time_Prediction.csv")
df.head()


## Dataset Information

In [ ]:

print(df.shape)
print(df.info())
print(df.isnull().sum())


## Descriptive Statistics

In [ ]:

df.describe(include='all')


## Correlation Analysis

In [ ]:

numeric_df = df.select_dtypes(include=np.number)

plt.figure(figsize=(10,6))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()


## Outlier Detection

In [ ]:

for col in numeric_df.columns:
    plt.figure(figsize=(6,3))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot - {col}")
    plt.show()


## Feature Engineering

In [ ]:

df['Rush_Hour'] = df['Order_Time'].isin(['Morning','Evening']).astype(int)

df['Delivery_Status'] = np.where(
    df['Delivery_Time'] > df['Delivery_Time'].median(),
    'Delayed',
    'Fast'
)

df.head()


## Linear Regression

In [ ]:

X = df.drop(columns=['Order_ID','Customer_Location',
                     'Restaurant_Location','Delivery_Time',
                     'Delivery_Status'])

y = df['Delivery_Time']

categorical_cols = X.select_dtypes(include='object').columns
numeric_cols = X.select_dtypes(exclude='object').columns

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

linear_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

linear_model.fit(X_train, y_train)

y_pred = linear_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MSE:", mse)
print("MAE:", mae)
print("R2:", r2)


## Logistic Regression

In [ ]:

X_cls = df.drop(columns=['Order_ID','Customer_Location',
                         'Restaurant_Location','Delivery_Time',
                         'Delivery_Status'])

y_cls = (df['Delivery_Status'] == 'Delayed').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42
)

logistic_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

logistic_model.fit(X_train, y_train)

y_pred = logistic_model.predict(X_test)
y_prob = logistic_model.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))


## Confusion Matrix and ROC Curve

In [ ]:

cm = confusion_matrix(y_test, y_pred)

ConfusionMatrixDisplay(cm).plot()
plt.show()

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1],[0,1],'--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()



# Final Report

## Dataset Preprocessing
- Missing value handling
- Encoding categorical variables
- Feature scaling

## Models
1. Linear Regression for delivery time prediction.
2. Logistic Regression for Fast vs Delayed classification.

## Evaluation
- MSE, MAE, R²
- Accuracy, Precision, Recall, F1-score
- Confusion Matrix and ROC Curve

## Actionable Insights
- Optimize delivery routes.
- Increase staffing during rush hours.
- Train delivery personnel.
- Use traffic-aware dispatching.
